In [191]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns


# Hypothesis testing: Chi-Square Test within the Eniac case study

In this notebook we perform a chi-square test with the data from the Eniac case study, applying a post-hoc correction to perform pairwise tests and find the true winner.

## 1.&nbsp;State the Null Hypothesis and the Alternative Hypothesis.

<font color="green">H0: There is no significant difference between the versions A, B, C, and D.<br><br>
</font>

HA: There is at least 1 version that is significantly different to the other versions.

## 2.&nbsp; Select an appropriate significance level alpha ($\alpha$).

It was decided that a relatively high alpha was acceptable in this case

In [192]:
# 90% confidence level

alpha = 0.1

## 3.&nbsp; Collect data that is random and independent

The important pieces of information (clicks on each element of interest & visits on each page) are scattered around. Let's collect them. Where are the .csv files? 🥸

In [193]:
# read all csv files and create a dictionary for later use
# make it usable for GitHub

df_a, df_b, df_c, df_d = (
    pd.read_csv(
        f"https://raw.githubusercontent.com/FlorianHast/Eniac-ABTest/main/data/raw/eniac_{i}.csv"
    )
    for i in ["a", "b", "c", "d"]
)

dfs = {
    "a": df_a,
    "b": df_b,
    "c": df_c,
    "d": df_d
}

In [194]:
# create a info DataFrame from the Snapshot information of each table: string manipulation and dictionary creation

info = {}

for key, df in dfs.items():
    row0 = df["Snapshot information"][0].split("•")
    row1 = df["Snapshot information"][1].split("•")

    visits, clicks = [
        int(x.split()[0])
        for x in row1[2].split(",")
    ]

    info[key] = {
        "version": row0[0].strip(),
        "url": row0[1].strip(),
        "created": row1[0].strip(),
        "duration": row1[1].strip(),
        "visits": visits,
        "total clicks": clicks,
    }

info_df = pd.DataFrame.from_dict(info, orient="index")
info_df['version'] = ["a", "b", "c", "d"]

In [195]:
info_df

,version,url,created,duration,visits,total clicks
a,a,https://eniac.com/index-a.php,created 2021-09-14,14 days 0 hours 34 mins,25326,23174
b,b,https://eniac.com/index-b.php,created 2021-10-27,14 days 0 hours 34 mins,24747,22407
c,c,https://eniac.com/index-c.php,created 2021-10-27,14 days 0 hours 34 mins,24876,23031
d,d,https://eniac.com/index-d.php,created 2021-10-27,14 days 0 hours 34 mins,25233,22743


In [196]:
# create a DataFrame from the clicks corresponding the 'SHOP NOW'/'SEE DEALS' button for all versions

clicks_list = []

for version, df in dfs.items():
    clicks = df.loc[
        df["Name"].isin(["SHOP NOW", "SEE DEALS"]),
        "No. clicks"
    ].iloc[0]

    clicks_list.append({
        "version": version,
        "clicks": clicks
    })

clicks_df = pd.DataFrame(clicks_list)

clicks_df

,version,clicks
0,a,512
1,b,281
2,c,527
3,d,193


In [197]:
# merge info_df and clicks_df

clicks_df = clicks_df.merge(
    info_df,
    on="version",
    how="left"
)

clicks_df

,version,clicks,url,created,duration,visits,total clicks
0,a,512,https://eniac.com/index-a.php,created 2021-09-14,14 days 0 hours 34 mins,25326,23174
1,b,281,https://eniac.com/index-b.php,created 2021-10-27,14 days 0 hours 34 mins,24747,22407
2,c,527,https://eniac.com/index-c.php,created 2021-10-27,14 days 0 hours 34 mins,24876,23031
3,d,193,https://eniac.com/index-d.php,created 2021-10-27,14 days 0 hours 34 mins,25233,22743


In [198]:
# create the no clicks column: no clicks = visits - clicks

clicks_df['no clicks'] = clicks_df['visits'] - clicks_df['clicks']
clicks_df

,version,clicks,url,created,duration,visits,total clicks,no clicks
0,a,512,https://eniac.com/index-a.php,created 2021-09-14,14 days 0 hours 34 mins,25326,23174,24814
1,b,281,https://eniac.com/index-b.php,created 2021-10-27,14 days 0 hours 34 mins,24747,22407,24466
2,c,527,https://eniac.com/index-c.php,created 2021-10-27,14 days 0 hours 34 mins,24876,23031,24349
3,d,193,https://eniac.com/index-d.php,created 2021-10-27,14 days 0 hours 34 mins,25233,22743,25040


In [199]:
# create result DataFrame for chi-square test: select only clicks and no clicks, transpose and set ["A","B","C","D"] as colums names

results_df = clicks_df[['clicks', 'no clicks']].T
results_df.columns=["A","B","C","D"]

results_df

,A,B,C,D
clicks,512,281,527,193
no clicks,24814,24466,24349,25040


## 4.&nbsp; Calculate the test result

In [200]:
from scipy.stats import chi2_contingency

chisq, pvalue, df, expected = chi2_contingency(results_df)

chisq, pvalue, df, expected

(np.float64(224.01877488058412),
 np.float64(2.7161216607868712e-48),
 3,
 array([[  382.48625502,   373.74189974,   375.69012397,   381.08172127],
        [24943.51374498, 24373.25810026, 24500.30987603, 24851.91827873]]))

## 5.&nbsp; Interpret the test result

In [201]:
if pvalue < alpha:
  print(f"Since p-value is SMALLER than alpha ({pvalue:.2e} < {alpha}), we must REJECT the H0!")
  print("Therfore, there is a SIGNIFICANT DIFFERENCE for at least one of the four versions!\nLet´s find out which!")
else:
  print(f"Since p-value is GREATER than alpha ({pvalue:.2e} > {alpha}), we can FAIL to reject the H0!")
  print("Therfore, there is a NO significant difference between the four versions!")

Since p-value is SMALLER than alpha (2.72e-48 < 0.1), we must REJECT the H0!
Therfore, there is a SIGNIFICANT DIFFERENCE for at least one of the four versions!
Let´s find out which!


## How do we decide who's the winner?

In [202]:
# Post Hoc Analysis

from scipy.stats import chi2_contingency
from itertools import combinations

versions = ['A', 'B', 'C', 'D']
pairs = list(combinations(versions, 2))
pairs

bonalpha = alpha/len(pairs)      # Bonferroni correction

print(f"Bonferroni alpha: {bonalpha:.4f}\n")

for v1, v2 in pairs:
  sub_table = results_df[[v1,v2,]]
  chisquare, p_val, dof, expected = chi2_contingency(sub_table)
  print(f'{v1} vs. {v2} -> {p_val:.4e}:',
        f'{v1} and {v2} differ significantly' if p_val < bonalpha else f'{v1} and {v2} show no significant difference')

Bonferroni alpha: 0.0167

A vs. B -> 2.6731e-15: A and B differ significantly
A vs. C -> 4.6484e-01: A and C show no significant difference
A vs. D -> 3.0809e-33: A and D differ significantly
B vs. C -> 6.9555e-18: B and C differ significantly
B vs. D -> 2.3573e-05: B and D differ significantly
C vs. D -> 6.4505e-37: C and D differ significantly


In [203]:
# Next Steps: Conversion rate to see a relative increase/decrease:

ctr_a = results_df.loc['clicks', 'A'] / sum(results_df['A'])
ctr_b = results_df.loc['clicks', 'B'] / sum(results_df['B'])
ctr_c = results_df.loc['clicks', 'C'] / sum(results_df['C'])
ctr_d = results_df.loc['clicks', 'D'] / sum(results_df['D'])

ctrs = pd.DataFrame({'versions': ['A', 'B', 'C', 'D'], 'CTR': [ctr_a, ctr_b, ctr_c, ctr_d]})

ctrs.sort_values('CTR', ascending=False)

,versions,CTR
2,C,0.021185
0,A,0.020216
1,B,0.011355
3,D,0.007649
